In [ ]:
#@title LIF Neuron

# ============================================================
# Interactive Leaky Integrate-and-Fire Neuron
# Google Colab-compatible
#
# One presynaptic spike train drives one postsynaptic neuron.
# Change parameters, then click RUN SIMULATION.
# ============================================================

# If widgets do not display in Colab, uncomment these lines:
# !pip -q install ipywidgets
# from google.colab import output
# output.enable_custom_widget_manager()

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output


# ============================================================
# 1. Generate the presynaptic spike train
# ============================================================

def generate_spike_train(
    rate_hz,
    duration_ms,
    dt_ms,
    train_type,
    seed
):
    """
    Generate a single presynaptic spike train.

    Parameters
    ----------
    rate_hz : float
        Mean presynaptic firing rate in Hz.

    duration_ms : float
        Total simulation duration in milliseconds.

    dt_ms : float
        Simulation time step in milliseconds.

    train_type : str
        Either "Fixed ISI" or "Poisson".

    seed : int
        Random-number seed.

    Returns
    -------
    spikes : numpy.ndarray
        Binary array containing 1 at presynaptic spike times.

    spike_times_ms : numpy.ndarray
        Presynaptic spike times in milliseconds.
    """

    n_steps = int(np.round(duration_ms / dt_ms))
    spikes = np.zeros(n_steps, dtype=float)

    if rate_hz <= 0:
        return spikes, np.array([])

    rng = np.random.default_rng(seed)

    if train_type == "Fixed ISI":

        # Interspike interval in milliseconds
        isi_ms = 1000.0 / rate_hz

        # Randomize the phase so the first spike does not always occur at t = 0
        first_spike_ms = rng.uniform(0.0, isi_ms)

        spike_times_ms = np.arange(
            first_spike_ms,
            duration_ms,
            isi_ms
        )

        spike_indices = np.round(
            spike_times_ms / dt_ms
        ).astype(int)

        spike_indices = spike_indices[
            (spike_indices >= 0) &
            (spike_indices < n_steps)
        ]

        spikes[spike_indices] = 1.0
        spike_times_ms = spike_indices * dt_ms

    elif train_type == "Poisson":

        # Probability of a spike in one time bin
        spike_probability = rate_hz * dt_ms / 1000.0

        if spike_probability > 0.1:
            print(
                "Warning: rate × dt is relatively large. "
                "Use a smaller dt for a more accurate Poisson process."
            )

        spikes = (
            rng.random(n_steps)
            < np.clip(spike_probability, 0.0, 1.0)
        ).astype(float)

        spike_times_ms = np.flatnonzero(spikes) * dt_ms

    else:
        raise ValueError(
            "train_type must be 'Fixed ISI' or 'Poisson'."
        )

    return spikes, spike_times_ms


# ============================================================
# 2. Simulate the leaky integrate-and-fire neuron
# ============================================================

def simulate_lif(
    rate_hz=40.0,
    train_type="Poisson",
    synaptic_weight_mv=5.0,
    tau_syn_ms=10.0,
    tau_membrane_ms=20.0,
    resting_potential_mv=-65.0,
    threshold_mv=-50.0,
    reset_potential_mv=-65.0,
    refractory_period_ms=2.0,
    duration_ms=1000.0,
    dt_ms=0.1,
    seed=1
):
    """
    Simulate a leaky integrate-and-fire neuron.

    Synaptic dynamics
    -----------------
    ds/dt = -s / tau_syn

    At every presynaptic spike:
    s <- s + synaptic_weight

    Membrane dynamics
    -----------------
    dV/dt = (V_rest - V + s) / tau_membrane

    Positive synaptic weights are excitatory.
    Negative synaptic weights are inhibitory.
    """

    # --------------------------------------------------------
    # Validate parameters
    # --------------------------------------------------------

    if dt_ms <= 0:
        raise ValueError("dt_ms must be positive.")

    if duration_ms <= 0:
        raise ValueError("duration_ms must be positive.")

    if tau_syn_ms <= 0:
        raise ValueError("tau_syn_ms must be positive.")

    if tau_membrane_ms <= 0:
        raise ValueError("tau_membrane_ms must be positive.")

    if threshold_mv <= reset_potential_mv:
        raise ValueError(
            "Threshold must be above the reset potential."
        )

    # --------------------------------------------------------
    # Create time axis and input spike train
    # --------------------------------------------------------

    n_steps = int(np.round(duration_ms / dt_ms))
    time_ms = np.arange(n_steps) * dt_ms

    input_spikes, input_spike_times_ms = generate_spike_train(
        rate_hz=rate_hz,
        duration_ms=duration_ms,
        dt_ms=dt_ms,
        train_type=train_type,
        seed=seed
    )

    # --------------------------------------------------------
    # Allocate output arrays
    # --------------------------------------------------------

    synaptic_drive_mv = np.zeros(n_steps)
    voltage_mv = np.full(
        n_steps,
        resting_potential_mv,
        dtype=float
    )

    output_spikes = np.zeros(n_steps, dtype=float)

    # --------------------------------------------------------
    # Initialize neuron state
    # --------------------------------------------------------

    synaptic_state = 0.0
    voltage = resting_potential_mv

    refractory_steps = int(
        np.round(refractory_period_ms / dt_ms)
    )

    refractory_steps_remaining = 0

    # Exact exponential decay per simulation step
    synaptic_decay_factor = np.exp(
        -dt_ms / tau_syn_ms
    )

    # --------------------------------------------------------
    # Simulation loop
    # --------------------------------------------------------

    for i in range(1, n_steps):

        # Decay the existing postsynaptic potential
        synaptic_state *= synaptic_decay_factor

        # Add the signed synaptic weight at an input spike
        if input_spikes[i] == 1:
            synaptic_state += synaptic_weight_mv

        synaptic_drive_mv[i] = synaptic_state

        # Hold voltage at reset during refractory period
        if refractory_steps_remaining > 0:

            voltage = reset_potential_mv
            refractory_steps_remaining -= 1

        else:

            # Leaky integration
            dv = (
                resting_potential_mv
                - voltage
                + synaptic_state
            ) * dt_ms / tau_membrane_ms

            voltage += dv

            # Spike threshold crossing
            if voltage >= threshold_mv:

                output_spikes[i] = 1.0

                # Store threshold at the spike time for plotting
                voltage_mv[i] = threshold_mv

                voltage = reset_potential_mv
                refractory_steps_remaining = refractory_steps

                continue

        voltage_mv[i] = voltage

    output_spike_times_ms = (
        np.flatnonzero(output_spikes) * dt_ms
    )

    return {
        "time_ms": time_ms,
        "input_spikes": input_spikes,
        "input_spike_times_ms": input_spike_times_ms,
        "synaptic_drive_mv": synaptic_drive_mv,
        "voltage_mv": voltage_mv,
        "output_spikes": output_spikes,
        "output_spike_times_ms": output_spike_times_ms
    }


# ============================================================
# 3. Plot the simulation
# ============================================================

def plot_simulation(parameters):
    """
    Run the simulation and produce four plots.
    """

    result = simulate_lif(**parameters)

    time_ms = result["time_ms"]
    input_times = result["input_spike_times_ms"]
    output_times = result["output_spike_times_ms"]

    duration_seconds = (
        parameters["duration_ms"] / 1000.0
    )

    measured_input_rate = (
        len(input_times) / duration_seconds
    )

    measured_output_rate = (
        len(output_times) / duration_seconds
    )

    fig, axes = plt.subplots(
        4,
        1,
        figsize=(13, 10),
        sharex=True,
        gridspec_kw={
            "height_ratios": [1, 2, 3, 1]
        }
    )

    # --------------------------------------------------------
    # Plot 1: presynaptic spikes
    # --------------------------------------------------------

    axes[0].eventplot(
        input_times,
        lineoffsets=0.5,
        linelengths=0.8
    )

    axes[0].set_ylim(0, 1)
    axes[0].set_yticks([])
    axes[0].set_ylabel("Input")
    axes[0].grid(
        True,
        axis="x",
        alpha=0.25
    )

    axes[0].set_title(
        f"Presynaptic spike train: "
        f"{parameters['train_type']}, "
        f"requested rate = "
        f"{parameters['rate_hz']:.1f} Hz"
    )

    # --------------------------------------------------------
    # Plot 2: synaptic drive
    # --------------------------------------------------------

    axes[1].plot(
        time_ms,
        result["synaptic_drive_mv"],
        linewidth=1.4
    )

    axes[1].axhline(
        0,
        linewidth=0.8
    )

    axes[1].set_ylabel(
        "Synaptic\ndrive (mV)"
    )

    axes[1].grid(
        True,
        alpha=0.25
    )

    # --------------------------------------------------------
    # Plot 3: membrane voltage
    # --------------------------------------------------------

    axes[2].plot(
        time_ms,
        result["voltage_mv"],
        linewidth=1.4
    )

    axes[2].axhline(
        parameters["threshold_mv"],
        linestyle="--",
        linewidth=1.2,
        label=(
            f"Threshold = "
            f"{parameters['threshold_mv']:.1f} mV"
        )
    )

    axes[2].axhline(
        parameters["resting_potential_mv"],
        linestyle=":",
        linewidth=1.0,
        label=(
            f"Rest = "
            f"{parameters['resting_potential_mv']:.1f} mV"
        )
    )

    # Mark output spikes with vertical lines
    for spike_time in output_times:
        axes[2].axvline(
            spike_time,
            alpha=0.22,
            linewidth=0.8
        )

    axes[2].set_ylabel(
        "Membrane\npotential (mV)"
    )

    axes[2].legend(
        loc="upper right"
    )

    axes[2].grid(
        True,
        alpha=0.25
    )

    # --------------------------------------------------------
    # Plot 4: postsynaptic spikes
    # --------------------------------------------------------

    axes[3].eventplot(
        output_times,
        lineoffsets=0.5,
        linelengths=0.8
    )

    axes[3].set_ylim(0, 1)
    axes[3].set_yticks([])
    axes[3].set_ylabel("Output")
    axes[3].set_xlabel("Time (ms)")

    axes[3].grid(
        True,
        axis="x",
        alpha=0.25
    )

    # --------------------------------------------------------
    # Overall summary title
    # --------------------------------------------------------

    fig.suptitle(
        f"Input spikes: {len(input_times)} "
        f"({measured_input_rate:.1f} Hz measured)   |   "
        f"Output spikes: {len(output_times)} "
        f"({measured_output_rate:.1f} Hz)",
        fontsize=12,
        y=0.995
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# 4. Interactive widget controls
# ============================================================

widget_style = {
    "description_width": "180px"
}

widget_layout = widgets.Layout(
    width="540px"
)

rate_widget = widgets.FloatSlider(
    value=40.0,
    min=0.0,
    max=200.0,
    step=1.0,
    description="Input rate (Hz)",
    continuous_update=False,
    style=widget_style,
    layout=widget_layout
)

train_type_widget = widgets.Dropdown(
    options=[
        "Fixed ISI",
        "Poisson"
    ],
    value="Poisson",
    description="Input timing",
    style=widget_style,
    layout=widget_layout
)

synaptic_weight_widget = widgets.FloatSlider(
    value=5.0,
    min=-20.0,
    max=20.0,
    step=0.25,
    description="Synaptic weight (mV)",
    continuous_update=False,
    style=widget_style,
    layout=widget_layout
)

tau_syn_widget = widgets.FloatLogSlider(
    value=10.0,
    base=10,
    min=np.log10(1.0),
    max=np.log10(200.0),
    step=0.02,
    description="Synaptic decay τ (ms)",
    continuous_update=False,
    style=widget_style,
    layout=widget_layout,
    readout_format=".2f"
)

tau_membrane_widget = widgets.FloatLogSlider(
    value=20.0,
    base=10,
    min=np.log10(2.0),
    max=np.log10(200.0),
    step=0.02,
    description="Membrane τ (ms)",
    continuous_update=False,
    style=widget_style,
    layout=widget_layout,
    readout_format=".2f"
)

resting_potential_widget = widgets.FloatSlider(
    value=-65.0,
    min=-90.0,
    max=-40.0,
    step=1.0,
    description="Resting potential (mV)",
    continuous_update=False,
    style=widget_style,
    layout=widget_layout
)

threshold_widget = widgets.FloatSlider(
    value=-50.0,
    min=-75.0,
    max=-20.0,
    step=1.0,
    description="Threshold (mV)",
    continuous_update=False,
    style=widget_style,
    layout=widget_layout
)

reset_potential_widget = widgets.FloatSlider(
    value=-65.0,
    min=-90.0,
    max=-30.0,
    step=1.0,
    description="Reset potential (mV)",
    continuous_update=False,
    style=widget_style,
    layout=widget_layout
)

refractory_period_widget = widgets.FloatSlider(
    value=2.0,
    min=0.0,
    max=20.0,
    step=0.5,
    description="Refractory period (ms)",
    continuous_update=False,
    style=widget_style,
    layout=widget_layout
)

duration_widget = widgets.FloatSlider(
    value=1000.0,
    min=100.0,
    max=5000.0,
    step=100.0,
    description="Duration (ms)",
    continuous_update=False,
    style=widget_style,
    layout=widget_layout
)

dt_widget = widgets.Dropdown(
    options=[
        0.05,
        0.1,
        0.2,
        0.5,
        1.0
    ],
    value=0.1,
    description="Time step dt (ms)",
    style=widget_style,
    layout=widget_layout
)

seed_widget = widgets.IntSlider(
    value=1,
    min=0,
    max=100,
    step=1,
    description="Random seed",
    continuous_update=False,
    style=widget_style,
    layout=widget_layout
)


# ============================================================
# 5. RUN SIMULATION button
# ============================================================

run_button = widgets.Button(
    description="RUN SIMULATION",
    button_style="success",
    icon="play",
    layout=widgets.Layout(
        width="220px",
        height="45px"
    )
)

output_area = widgets.Output()


def get_current_parameters():
    """
    Read the current values from all widgets.
    """

    return {
        "rate_hz": rate_widget.value,
        "train_type": train_type_widget.value,
        "synaptic_weight_mv": synaptic_weight_widget.value,
        "tau_syn_ms": tau_syn_widget.value,
        "tau_membrane_ms": tau_membrane_widget.value,
        "resting_potential_mv": resting_potential_widget.value,
        "threshold_mv": threshold_widget.value,
        "reset_potential_mv": reset_potential_widget.value,
        "refractory_period_ms": refractory_period_widget.value,
        "duration_ms": duration_widget.value,
        "dt_ms": dt_widget.value,
        "seed": seed_widget.value
    }


def run_button_clicked(button):
    """
    Run the simulation only when the button is clicked.
    """

    with output_area:

        clear_output(wait=True)

        parameters = get_current_parameters()

        try:
            plot_simulation(parameters)

        except ValueError as error:
            print(f"Parameter error: {error}")

        except Exception as error:
            print(f"Unexpected error: {error}")


run_button.on_click(run_button_clicked)


# ============================================================
# 6. Display the interface
# ============================================================

instructions = widgets.HTML(
    """
    <h3>Interactive Leaky Integrate-and-Fire Neuron</h3>

    <p>
    Adjust the parameters below, then click
    <b>RUN SIMULATION</b>.
    Changing a slider does not automatically rerun the model.
    </p>

    <p>
    A positive synaptic weight is excitatory.
    A negative synaptic weight is inhibitory.
    </p>
    """
)

input_controls = widgets.VBox([
    widgets.HTML("<b>Presynaptic input</b>"),
    rate_widget,
    train_type_widget,
    seed_widget
])

synapse_controls = widgets.VBox([
    widgets.HTML("<b>Synapse</b>"),
    synaptic_weight_widget,
    tau_syn_widget
])

neuron_controls = widgets.VBox([
    widgets.HTML("<b>Postsynaptic neuron</b>"),
    tau_membrane_widget,
    resting_potential_widget,
    threshold_widget,
    reset_potential_widget,
    refractory_period_widget
])

simulation_controls = widgets.VBox([
    widgets.HTML("<b>Simulation</b>"),
    duration_widget,
    dt_widget
])

control_panel = widgets.VBox([
    instructions,
    input_controls,
    synapse_controls,
    neuron_controls,
    simulation_controls,
    run_button
])

display(
    control_panel,
    output_area
)

# Run once initially so the notebook starts with a plot
run_button_clicked(None)

Output()